In [ ]:
import os
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from xgboost.spark import SparkXGBClassifier

In [2]:
spark = SparkSession.builder \
    .appName("ai_ensembler") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [3]:
def get_base_data(df):
    """Creates the base targets and machine state without heavy windowing."""
    for n in range(1, 41):
        target = f"n_{n}"
        df = df.withColumn(target, F.when(F.array_contains(F.array("B1","B2","B3","B4","B5","B6"), n), 1).otherwise(0))
    
    df = df.withColumn("row_idx", F.row_number().over(Window.orderBy("Date")))
    df = df.withColumn("evens", sum([(F.col(f"B{i}") % 2 == 0).cast("int") for i in range(1, 4)]))
    return df.na.fill(0)


In [4]:
def train_ball_logic(df, ball_num):
    """Integrated Logic: Momentum + Gap + Freq + Ensemble Models."""
    w_gap = Window.orderBy("Date").rowsBetween(Window.unboundedPreceding, -1)
    w_freq = Window.orderBy("Date").rowsBetween(-20, -1)
    w_mom = Window.orderBy("Date").rowsBetween(-5, -1)
    
    target = f"n_{ball_num}"
    
    # Feature Engineering (On-the-fly to save RAM)
    temp_df = df.withColumn(f"freq_{ball_num}", F.sum(F.col(target)).over(w_freq)) \
                .withColumn(f"mom_{ball_num}", F.sum(F.col(target)).over(w_mom)) \
                .withColumn(f"gap_{ball_num}", F.col("row_idx") - F.last(F.when(F.col(target) == 1, F.col("row_idx")), True).over(w_gap)) \
                .select("Date", target, f"freq_{ball_num}", f"mom_{ball_num}", f"gap_{ball_num}", "evens") \
                .na.fill(0)
    
    features = [f"gap_{ball_num}", f"freq_{ball_num}", f"mom_{ball_num}", "evens"]
    assembler = VectorAssembler(inputCols=features, outputCol="features")
    data_vec = assembler.transform(temp_df)
    
    # Ensemble Models (XGBoost + Random Forest)
    xgb = SparkXGBClassifier(n_estimators=30, max_depth=3, label_col=target).fit(data_vec)
    rf = RandomForestClassifier(numTrees=15, maxDepth=4, labelCol=target).fit(data_vec)
    
    # Probability Extraction for the most recent draw
    latest = data_vec.orderBy(F.desc("Date")).limit(1)
    p_xgb = xgb.transform(latest).select("probability").first()[0].toArray()[1]
    p_rf = rf.transform(latest).select("probability").first()[0].toArray()[1]
    
    return (p_xgb * 0.7) + (p_rf * 0.3)


In [5]:
def render_dashboard(scores, stability_val):
    """Displays the AI report and the 7-Line Wheel."""
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_8 = [ball for ball, _ in ranked[:8]]
    
    # Status Logic
    if stability_val >= 0.7: status, icon = "LOCKED", "🔒"
    elif stability_val >= 0.5: status, icon = "SYNCING", "🔄"
    else: status, icon = "CHAOTIC", "⚠️"
    
    report = []
    report.append("╔" + "═"*58 + "╗")
    report.append(f"║ {'AI MOMENTUM DASHBOARD':^56} ║")
    report.append("╠" + "═"*58 + "╣")
    report.append(f"║ STATUS: {status:<15} {icon} | STABILITY: {stability_val*100:>5.1f}%  ║")
    report.append("╟" + "─"*58 + "╢")
    
    for i, (ball, score) in enumerate(ranked[:10], 1):
        bar = "█" * int(score * 100)
        report.append(f"║ {i:>2}. Ball {ball:<2} | {score:.4f} | {bar:<25} ║")

    report.append("╠" + "═"*58 + "╣")
    report.append(f"║ {'7-LINE ABBREVIATED WHEEL (TOP 8)':^56} ║")
    report.append("╠" + "═"*58 + "╣")
    
    patterns = [[0,1,2,3,4,5], [0,1,2,6,7,3], [0,1,4,5,6,7], [2,3,4,5,6,7], [0,2,4,6,1,3], [1,3,5,7,0,2], [0,3,4,7,1,5]]
    for idx, p in enumerate(patterns, 1):
        line = sorted([top_8[i] for i in p])
        report.append(f"║ Line {idx}: {str(line):<48} ║")
    
    report.append("╚" + "═"*58 + "╝")
    
    final_output = "\n".join(report)
    print(final_output)
    
    with open("momentum_report.txt", "w", encoding="utf-8") as f:
        f.write(final_output)


In [6]:
def generate_4_line_sniper(core, primary_scores):
    # Rank all balls to fill gaps if needed
    ranked_all = sorted(primary_scores, key=primary_scores.get, reverse=True)
    base_core = sorted(core[:6]) 
    backups = [b for b in ranked_all if b not in base_core]

    def clean_line(raw_list):
        unique_line = []
        for b in raw_list:
            if b not in unique_line: unique_line.append(b)
        for b in backups:
            if len(unique_line) == 6: break
            if b not in unique_line: unique_line.append(b)
        return sorted(unique_line)

    # 4-Line Matrix Logic
    line1 = clean_line(base_core)
    line2 = clean_line([base_core[0], 4, base_core[1], base_core[2], 18, base_core[4]])
    line3 = clean_line([base_core[1], 8, base_core[2], base_core[3], base_core[4], base_core[5]])
    line4 = clean_line([4, 8, 12, 18, 24, 36]) # High-rank coverage line

    print("\n" + "🎯"*20)
    print("      SNIPER TICKET: 4-LINE OPTIMIZED")
    print("🎯"*20)
    for i, line in enumerate([line1, line2, line3, line4], 1):
        print(f"LINE {i}: {line}")
    print("🎯"*20)



In [7]:
def check_machine_bias_and_sync(df, sniper_core):
    """
    Analyzes the last 30 draws to see if the AI's 'Even-Heavy' 
    prediction matches the current machine behavior.
    """
    # Get last 30 draws for a solid statistical sample
    recent_30 = df.orderBy(F.desc("Date")).limit(30).collect()
    
    even_count = 0
    high_count = 0 # Numbers 21-40
    total_balls = 30 * 6
    hit_map = {} 

    for row in recent_30:
        draw = [row[f'B{i}'] for i in range(1, 7)]
        for b in draw:
            hit_map[b] = hit_map.get(b, 0) + 1
            if b % 2 == 0: even_count += 1
            if b > 20: high_count += 1
            
    # Calculate Bias Percentages
    even_pct = (even_count / total_balls) * 100
    high_pct = (high_count / total_balls) * 100
    
    # Calculate "Sync" - How many of your sniper balls are actually 'Hot'
    sorted_hits = sorted(hit_map.items(), key=lambda x: x[1], reverse=True)
    hot_balls = [ball for ball, count in sorted_hits[:10]]
    sync_score = len(set(sniper_core).intersection(set(hot_balls)))

    print("\n" + "📊"*20)
    print(f"      MACHINE BIAS & HEATMAP REPORT")
    print("📊"*20)
    print(f"CURRENT EVEN BIAS: {even_pct:.1f}%")
    print(f"CURRENT HIGH BIAS: {high_pct:.1f}%")
    print("-" * 40)
    print(f"🔥 HOT BALL SYNC: {sync_score}/6")
    print("(How many Sniper balls are in the Top 10 'Hot' list)")
    print("-" * 40)
    
    print("🔝 TOP 5 HOTTEST BALLS RECENTLY:")
    for ball, count in sorted_hits[:5]:
        print(f"Ball {ball}: {count} hits in last 30 draws")
        
    return sync_score


In [8]:
def run_backtest(df, primary_scores, top_8_now, top_8_then):
    """
    Automates the identification of the 6 Stable Anchors 
    and checks their performance over the last 10 draws.
    """
    # 1. Identify the 6 Stable Anchors
    sniper_core = sorted(list(top_8_now.intersection(top_8_then)))
    
    print("\n" + "🔍"*20)
    print(f"      STABILITY AUDIT: {len(sniper_core)}/8 BALLS LOCKED")
    print(f"      ANCHORS: {sniper_core}")
    print("🔍"*20)

    # 2. Historical Hit Check (Last 10 Draws)
    # Collect actual winning numbers for the last 10 draws
    history = df.orderBy(F.desc("Date")).limit(10).collect()
    
    print("\n--- LAST 10 DRAW BACKTEST ---")
    total_hits = 0
    for row in history:
        draw_date = str(row['Date'])[:10]
        # Collect actual winning numbers from columns B1-B6
        actual_numbers = {row[f'B{i}'] for i in range(1, 7)}
        
        # Count how many of our 6 Sniper Anchors hit in this draw
        hits = set(sniper_core).intersection(actual_numbers)
        total_hits += len(hits)
        
        status = "🎯 HIT!" if len(hits) > 0 else "❌ Miss"
        print(f"Date: {draw_date} | Actual: {sorted(list(actual_numbers))} | Sniper Hits: {len(hits)} {status}")

    avg_hit = total_hits / 10
    print("-" * 40)
    print(f"BACKTEST SUMMARY: Avg {avg_hit:.1f} Sniper Hits per draw.")
    print("-" * 40)
    
    return sniper_core


In [9]:
def render_sniper_wheel(core):
    """
    Generates a 3-line high-density wheel for 6 numbers.
    Ensures that if 4 of the 6 numbers hit, you have a high chance of a 4-hit line.
    """
    if len(core) < 6:
        print("⚠️ Not enough stable balls for a Sniper Wheel. Using top 6 from primary scores instead.")
        core = sorted(sorted(primary_scores, key=primary_scores.get, reverse=True)[:6])

    # Balanced 3-line pattern for 6 numbers
    # Each line uses a different combination of the core
    sniper_patterns = [
        [0, 1, 2, 3, 4, 5], # Line 1: The Full Core (The "Pure" Play)
        [0, 1, 2, 3, 4, 5], # Placeholder: In a 6-number lotto, 1 line covers all 6.
    ]
    
    # If your lotto requires 6 numbers, 6 balls fit into exactly 1 line.
    # To create a '3-line Sniper', we mix in the next 2 strongest candidates.
    next_best = sorted(primary_scores, key=primary_scores.get, reverse=True)[6:8]
    extended_core = core + next_best
    
    # Mathematical 3-line spread for 8 numbers (Sniper version)
    patterns = [
        [0, 1, 2, 3, 4, 5], # Focus on the 6 Stable Anchors
        [0, 1, 2, 4, 6, 7], # Mix Anchors with 2 "New" Momentum balls
        [2, 3, 4, 5, 6, 7]  # Mix Anchors with 2 "New" Momentum balls
    ]

    print("\n" + "🎯"*20)
    print(f"      SNIPER WHEEL (STABLE CORE: {core})")
    print("🎯"*20)
    for i, p in enumerate(patterns, 1):
        line = sorted([extended_core[idx] for idx in p])
        print(f"SNIPER LINE {i}: {line}")
    print("🎯"*20)

In [10]:
print("📡 Step 1/3: Ingesting Data...")
raw = spark.read.csv("work/dataset/randomness_latest.csv", header=True, inferSchema=True)
base_data = get_base_data(raw)

print("📡 Step 2/3: Training Primary Models (Balls 1-40)...")
primary_scores = {}
for b in range(1, 41):
    primary_scores[b] = train_ball_logic(base_data, b)
    if b % 10 == 0: print(f"   [Progress: {b}/40]")

print("📡 Step 3/3: Running Forward Test (Stability Check)...")
# Get data excluding the very last draw
last_draw_date = raw.select(F.max("Date")).collect()[0][0]
n_minus_1_data = get_base_data(raw.filter(F.col("Date") < last_draw_date))

# Re-test top 10 candidates only to save memory
top_10 = sorted(primary_scores, key=primary_scores.get, reverse=True)[:10]
stable_scores = {b: train_ball_logic(n_minus_1_data, b) for b in top_10}

# Calculate Overlap (Stability)
top_8_now = set(sorted(primary_scores, key=primary_scores.get, reverse=True)[:8])
top_8_then = set(sorted(stable_scores, key=stable_scores.get, reverse=True)[:8])
stability = len(top_8_now & top_8_then) / 8

# Output Final Results
render_dashboard(primary_scores, stability)
print(f"The 6 Stable Anchors are: {top_8_now.intersection(top_8_then)}")
print(f"STABLE ANCHORS: {sorted(list(top_8_now.intersection(top_8_then)))}")


sniper_core = sorted(list(top_8_now.intersection(top_8_then)))
render_sniper_wheel(sniper_core)

sniper_core = run_backtest(raw, primary_scores, top_8_now, top_8_then)
sync_rating = check_machine_bias_and_sync(raw, sniper_core)
generate_4_line_sniper(sniper_core, primary_scores)


📡 Step 1/3: Ingesting Data...
📡 Step 2/3: Training Primary Models (Balls 1-40)...


2026-02-28 19:34:11,837 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:34:20,589 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:34:31,440 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:34:37,265 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:34:44,354 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_e

   [Progress: 10/40]


2026-02-28 19:36:12,917 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:36:18,571 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:36:24,785 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:36:30,390 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:36:38,553 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_e

   [Progress: 20/40]


...

   [Progress: 30/40]


2026-02-28 19:40:02,566 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:40:07,754 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:40:12,305 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:40:17,894 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:40:24,013 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_e

   [Progress: 40/40]
📡 Step 3/3: Running Forward Test (Stability Check)...


2026-02-28 19:42:03,995 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:42:10,258 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:42:14,472 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 30}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-28 19:42:19,994 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-02-28 19:42:26,121 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_e

╔══════════════════════════════════════════════════════════╗
║                  AI MOMENTUM DASHBOARD                   ║
╠══════════════════════════════════════════════════════════╣
║ STATUS: LOCKED          🔒 | STABILITY:  75.0%  ║
╟──────────────────────────────────────────────────────────╢
║  1. Ball 4  | 0.2670 | ██████████████████████████ ║
║  2. Ball 10 | 0.2576 | █████████████████████████ ║
║  3. Ball 12 | 0.2554 | █████████████████████████ ║
║  4. Ball 24 | 0.2504 | █████████████████████████ ║
║  5. Ball 22 | 0.2469 | ████████████████████████  ║
║  6. Ball 6  | 0.2353 | ███████████████████████   ║
║  7. Ball 2  | 0.2270 | ██████████████████████    ║
║  8. Ball 36 | 0.2219 | ██████████████████████    ║
║  9. Ball 18 | 0.2140 | █████████████████████     ║
║ 10. Ball 32 | 0.2112 | █████████████████████     ║
╠══════════════════════════════════════════════════════════╣
║             7-LINE ABBREVIATED WHEEL (TOP 8)             ║
╠═══════════════════════════════════════════════════